<a href="https://colab.research.google.com/github/ahmedkhanpakistani5-hash/-intern-feedback-to-understand-positive-and-negative-sentiments./blob/main/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedkhanpakistani5-hash/ahmad-flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

In [ ]:
!pip install transformers

In [ ]:
import numpy as np
import pandas as pd
from itertools import islice
from datasets import load_dataset
from transformers import pipeline

NEEDED = ["report_date", "client_hash_id", "content_hash_id", "gsc_data_available",
          "gsc_impressions", "gsc_clicks", "gsc_sum_position"]
MAX_ROWS = 1_000_000    # lower to 300_000 if Colab runs out of memory

have_df = "df" in globals() and all(c in df.columns for c in NEEDED)

if have_df:
    print("df already exists, so loading is skipped.")
else:
    ds = load_dataset("FlyRank/internship-warehouse",
                      "fact_content_daily_performance",
                      streaming=True, split="train")
    ds = ds.select_columns(NEEDED)
    df = pd.DataFrame(list(islice(ds, MAX_ROWS)))

print("rows:", len(df))
print("dates:", pd.to_datetime(df["report_date"]).min().date(), "to",
      pd.to_datetime(df["report_date"]).max().date())
print("days with data:", pd.to_datetime(df["report_date"]).nunique())
print("clients:", df["client_hash_id"].nunique())

span = (pd.to_datetime(df["report_date"]).max() - pd.to_datetime(df["report_date"]).min()).days
assert span >= 42, "df covers fewer than 42 days, so three 14-day windows will not fit. Raise MAX_ROWS or use the full-table load."

df already exists, so loading is skipped.
rows: 3852879
dates: 2025-01-27 to 2026-06-30
days with data: 514
clients: 22


## 1. Question

*The research question and the decision it supports.*

## 1. Question

**Research question:** Which pages are seen often in search but under-capture clicks for their
position, and does that gap persist, so that a title or meta review has something real to fix?

**Decision it supports:** A content team has limited review time. This work ranks pages so the
team reviews first the ones expected to keep losing the most clicks, each with one reason code
and one action label. The ranking is decision-support: it says where to look first, not what
caused the gap.

In [ ]:
print(f"{df['client_hash_id'].nunique()} clients, "
      f"{df.groupby(['client_hash_id', 'content_hash_id']).ngroups:,} pages, "
      f"{len(df):,} daily rows")

8 clients, 17,897 pages, 1,000,000 daily rows


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*
## 2. Data

- **Release / table:** FlyRank ML Internship warehouse, table `fact_content_daily_performance`
  (daily search-console rows per page). Built on the FlyRank ML Internship dataset (https://flyrank.ai).
- **Date range:** ____ to ____ (____ days with data).
- **Windows (14 days each):** window 1 = ____, window 2 = ____, window 3 = ____.
- **Rows used:** ____ | **Pages:** ____ | **Clients:** ____ (IDs are hashed and never shown).
- **Excluded and why:** rows where search data was not available (____ rows); pages with fewer
  than 50 impressions in a window, because their CTR is too noisy to judge.
- **Public-safe:** no client names, domains, URLs or queries appear in this paper.

In [ ]:
base = df[df["gsc_data_available"].astype(bool)].copy()
base["report_date"] = pd.to_datetime(base["report_date"]).dt.normalize()
for c in ["gsc_impressions", "gsc_clicks", "gsc_sum_position"]:
    base[c] = pd.to_numeric(base[c], errors="coerce").fillna(0)

key = ["client_hash_id", "content_hash_id"]

# ---------- settings used by the whole notebook ----------
WIN = 14            # days per window
MIN_IMPR = 50       # minimum impressions in a window for a page to count
LABEL_RATIO = 0.8   # label = CTR below 80% of normal for its position
POS_BINS = [-np.inf, 3, 5, 10, 20, np.inf]
POS_LABELS = ["1-3", "3-5", "5-10", "10-20", "20+"]

start = base["report_date"].min()
last = base["report_date"].max()
d = pd.Timedelta(days=WIN)
W1 = (start, start + d)             # window 1: features for the training rows
W2 = (start + d, start + 2 * d)     # window 2: training labels AND features for the test rows
W3 = (start + 2 * d, start + 3 * d) # window 3: test labels
assert last >= W3[1] - pd.Timedelta(days=1), "Not enough days for three windows. Lower WIN."

def show(w):
    return f"{w[0].date()} to {(w[1] - pd.Timedelta(days=1)).date()}"

clients = np.array(sorted(base["client_hash_id"].unique()))
print("Date range:", start.date(), "to", last.date(), f"({base['report_date'].nunique()} days with data)")
print("Rows used:", len(base), "| rows excluded (no search data):", len(df) - len(base))
print("Pages:", base.groupby(key).ngroups, "| Clients:", len(clients), "(IDs are hashed and never shown)")
print("Window 1:", show(W1), "| rows:", int(((base['report_date'] >= W1[0]) & (base['report_date'] < W1[1])).sum()))
print("Window 2:", show(W2), "| rows:", int(((base['report_date'] >= W2[0]) & (base['report_date'] < W2[1])).sum()))
print("Window 3:", show(W3), "| rows:", int(((base['report_date'] >= W3[0]) & (base['report_date'] < W3[1])).sum()))

Date range: 2025-01-27 to 2025-06-26 (144 days with data)
Rows used: 1000000 | rows excluded (no search data): 0
Pages: 17897 | Clients: 8 (IDs are hashed and never shown)
Window 1: 2025-01-27 to 2025-02-09 | rows: 9845
Window 2: 2025-02-10 to 2025-02-23 | rows: 46495
Window 3: 2025-02-24 to 2025-03-09 | rows: 63288


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*
## 3. Methodology

- **Assumptions:** average position = summed position / impressions. "Normal CTR" for a position
  band = pooled clicks / pooled impressions of pages in that band, computed from the feature window only.
- **Features (feature window only):** impressions, CTR, average position, CTR ratio vs normal,
  missing clicks vs normal, days active, share of impressions on the busiest day, position swing.
- **Label (label window only):** label = 1 if, in the next 14 days, the page has at least 50
  impressions AND its CTR is below 80% of the normal CTR for its position band. Label rate: ____ (train), ____ (test).
- **Baseline:** the Week 4 rule: score = impressions x (normal CTR - actual CTR), clipped at 0 ("missing clicks").
- **Models:** logistic regression and gradient boosting. Gradient boosting was chosen as the main model before looking at test results.
- **Validation design:** time-aware (train on window 1 features and window 2 labels; test on window 2
  features and window 3 labels), and grouped by client (test clients never appear in training; ____ clients in total).
- **Leakage checks:** assertions in the notebook confirm the windows do not overlap, no label-window
  column is a feature, and normal CTR comes from the past window.

In [ ]:
def add_band(frame):
    frame["pos_band"] = pd.cut(frame["pos"], bins=POS_BINS, labels=POS_LABELS).astype(str)
    return frame

def agg_pages(rows):
    g = rows.groupby(key).agg(impr=("gsc_impressions", "sum"),
                              clicks=("gsc_clicks", "sum"),
                              sumpos=("gsc_sum_position", "sum")).reset_index()
    g = g[g["impr"] > 0].copy()
    g["ctr"] = g["clicks"] / g["impr"]
    g["pos"] = g["sumpos"] / g["impr"]
    return add_band(g)

def daily_extras(rows):
    r = rows[rows["gsc_impressions"] > 0].copy()
    r["day_pos"] = r["gsc_sum_position"] / r["gsc_impressions"]
    e = r.groupby(key).agg(days_active=("report_date", "nunique"),
                           max_day_impr=("gsc_impressions", "max"),
                           pos_min=("day_pos", "min"),
                           pos_max=("day_pos", "max")).reset_index()
    e["pos_swing"] = e["pos_max"] - e["pos_min"]
    return e[key + ["days_active", "max_day_impr", "pos_swing"]]

def normal_table(pages):
    # "normal" CTR per position band = pooled clicks / pooled impressions, from the FEATURE window only
    p = pages[pages["impr"] >= MIN_IMPR]
    t = p.groupby("pos_band").agg(c=("clicks", "sum"), i=("impr", "sum"))
    return (t["c"] / t["i"]).to_dict()

def build_dataset(f_start, f_end, l_end):
    """Features come from [f_start, f_end). The label comes from [f_end, l_end). l_end=None means no label."""
    feat_rows = base[(base["report_date"] >= f_start) & (base["report_date"] < f_end)]
    F = agg_pages(feat_rows).merge(daily_extras(feat_rows), on=key, how="left")
    normal = normal_table(F)
    F = F[F["impr"] >= MIN_IMPR].copy()
    F["expected_ctr"] = F["pos_band"].map(normal).astype(float)
    F.loc[F["expected_ctr"] <= 0, "expected_ctr"] = np.nan
    F["ctr_ratio"] = F["ctr"] / F["expected_ctr"]
    F["missing_clicks"] = (F["impr"] * (F["expected_ctr"] - F["ctr"]).clip(lower=0)).fillna(0)
    F["log_impr"] = np.log1p(F["impr"])
    F["top_day_share"] = F["max_day_impr"] / F["impr"]
    F["pos_band_code"] = F["pos_band"].map({b: i for i, b in enumerate(POS_LABELS)})
    if l_end is None:
        return F, normal
    lab_rows = base[(base["report_date"] >= f_end) & (base["report_date"] < l_end)]
    L = (agg_pages(lab_rows)[key + ["impr", "ctr", "pos", "pos_band"]]
         .rename(columns={"impr": "impr_l", "ctr": "ctr_l", "pos": "pos_l", "pos_band": "band_l"}))
    D = F.merge(L, on=key, how="inner")
    D = D[D["impr_l"] >= MIN_IMPR].copy()
    D["expected_l"] = D["band_l"].map(normal).astype(float)   # normal CTR is fixed from the PAST window
    D = D[D["expected_l"] > 0].copy()
    D["label"] = (D["ctr_l"] < LABEL_RATIO * D["expected_l"]).astype(int)
    return D, normal

FEATURES = ["impr", "log_impr", "ctr", "pos", "ctr_ratio", "missing_clicks",
            "days_active", "top_day_share", "pos_swing", "pos_band_code"]

# training rows: features from window 1, label from window 2
D_train_all, normal_w1 = build_dataset(W1[0], W1[1], W2[1])
# test rows: features from window 2, label from window 3 (later in time than every training label)
D_test_all, normal_w2 = build_dataset(W2[0], W2[1], W3[1])

print("train rows:", len(D_train_all), "| label rate:", round(D_train_all["label"].mean(), 3))
print("test rows: ", len(D_test_all), "| label rate:", round(D_test_all["label"].mean(), 3))
print("clients in train / test data:", D_train_all["client_hash_id"].nunique(), "/", D_test_all["client_hash_id"].nunique())

train rows: 369 | label rate: 0.531
test rows:  2803 | label rate: 0.634
clients in train / test data: 2 / 2


In [ ]:
GROUPED = len(clients) >= 6   # a client split needs enough clients to be meaningful

def get_split(seed):
    if not GROUPED:
        return D_train_all, D_test_all
    rng = np.random.RandomState(seed)
    perm = [str(c) for c in rng.permutation(clients)]
    k = int(round(0.7 * len(perm)))
    tr_c, te_c = perm[:k], perm[k:]
    return (D_train_all[D_train_all["client_hash_id"].isin(tr_c)],
            D_test_all[D_test_all["client_hash_id"].isin(te_c)])

if not GROUPED:
    print("NOTE: fewer than 6 clients, so the split is time-aware only (no client grouping). Say this in Limitations.")

# ---------- leakage checks ----------
assert W1[1] <= W2[0] and W2[1] <= W3[0], "windows overlap"
assert not any(c.endswith("_l") or c == "label" or "after" in c for c in FEATURES), "a label-window column is in FEATURES"
assert set(FEATURES).isdisjoint({"impr_l", "ctr_l", "pos_l", "band_l", "expected_l", "label"})
assert D_train_all["label"].nunique() == 2, "training labels have only one class. Change LABEL_RATIO or MIN_IMPR."
rate = D_train_all["label"].mean()
if rate < 0.02 or rate > 0.98:
    print("WARNING: label rate is extreme:", round(rate, 3), "- change LABEL_RATIO")

tr0, te0 = get_split(0)
print("main split | train rows:", len(tr0), "| test rows:", len(te0))
print("test clients are not in training:", (not GROUPED) or
      set(tr0["client_hash_id"]).isdisjoint(set(te0["client_hash_id"])))
print("CLEAN: features use only their window; labels use only the next window; normal CTR is fixed from the past window.")

main split | train rows: 369 | test rows: 0
test clients are not in training: True
CLEAN: features use only their window; labels use only the next window; normal CTR is fixed from the past window.


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 4. Results (vs baseline)

All methods are scored on the same test pages: ____ pages from clients not seen in training,
in a later time window. The base rate (share of test pages with the label) is ____.

(Paste the results table here.)

- **Precision at top 50:** baseline ____, boosting ____, boosting x missing clicks ____, random ____.
- **Versus the baseline:** the difference in precision at top 50 is ____ with a 95% bootstrap
  interval of [____, ____]. If that interval includes 0, the model is not clearly better than the plain rule.
- **Honest reading:** ____ (say plainly whether the model beat the rule, by how much, and how uncertain it is).

In [ ]:
print("clients in base:", len(clients), "| pages in base:", base.groupby(key).ngroups)
print("MIN_IMPR:", MIN_IMPR, "| WIN:", WIN, "| GROUPED:", GROUPED)
print("train rows (all):", len(D_train_all), "| test rows (all):", len(D_test_all))
print("clients with train rows:", D_train_all["client_hash_id"].nunique(),
      "| with test rows:", D_test_all["client_hash_id"].nunique())
print("rows per client (top 5, no IDs shown):",
      base.groupby("client_hash_id").size().sort_values(ascending=False).head(5).tolist())

clients in base: 8 | pages in base: 17897
MIN_IMPR: 50 | WIN: 14 | GROUPED: True
train rows (all): 369 | test rows (all): 2803
clients with train rows: 2 | with test rows: 2
rows per client (top 5, no IDs shown): [661180, 300603, 18422, 17023, 2414]


In [ ]:
import pandas as pd
import numpy as np
from datasets import load_dataset

NEEDED = ["report_date", "client_hash_id", "content_hash_id", "gsc_data_available",
          "gsc_impressions", "gsc_clicks", "gsc_sum_position"]
KEEP_ONE_IN = 3          # keep about 1/3 of clients. Use 2 for more data, 5 if memory is tight.
BATCH = 50_000

ds = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance",
                  streaming=True, split="train").select_columns(NEEDED)

kept, seen = [], 0
for batch in ds.iter(batch_size=BATCH):
    b = pd.DataFrame(batch)
    seen += len(b)
    h = pd.util.hash_pandas_object(b["client_hash_id"].astype(str), index=False).values
    b = b[(h % KEEP_ONE_IN) == 0]
    b = b[b["gsc_data_available"].astype(bool)]
    if len(b):
        kept.append(b)
    if (seen // BATCH) % 40 == 0:
        print(f"scanned {seen:,} rows, kept {sum(len(k) for k in kept):,}")

df = pd.concat(kept, ignore_index=True)
del kept

dts = pd.to_datetime(df["report_date"])
print("\nscanned rows:", f"{seen:,}", "| kept rows:", f"{len(df):,}")
print("dates:", dts.min().date(), "to", dts.max().date(), "| clients:", df["client_hash_id"].nunique())
assert df["client_hash_id"].nunique() >= 6, "fewer than 6 clients kept. Set KEEP_ONE_IN = 2 and rerun."
assert (dts.max() - dts.min()).days >= 30, "date range too short."

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

scanned 2,000,000 rows, kept 84,256
scanned 4,000,000 rows, kept 191,911
scanned 6,000,000 rows, kept 241,023
scanned 8,000,000 rows, kept 283,390
scanned 10,000,000 rows, kept 399,453
scanned 12,000,000 rows, kept 489,428
scanned 14,000,000 rows, kept 606,321
scanned 16,000,000 rows, kept 718,169
scanned 18,000,000 rows, kept 815,296
scanned 20,000,000 rows, kept 916,625
scanned 22,000,000 rows, kept 1,020,272
scanned 24,000,000 rows, kept 1,121,226
scanned 26,000,000 rows, kept 1,212,390
scanned 28,000,000 rows, kept 1,326,108
scanned 30,000,000 rows, kept 1,429,755
scanned 32,000,000 rows, kept 1,541,341
scanned 34,000,000 rows, kept 1,646,298
scanned 36,000,000 rows, kept 1,754,827
scanned 38,000,000 rows, kept 1,841,553
scanned 40,000,000 rows, kept 1,952,885
scanned 42,000,000 rows, kept 2,037,478
scanned 44,000,000 rows, kept 2,146,064
scanned 46,000,000 rows, kept 2,239,428
scanned 48,000,000 rows, kept 2,340,519
scanned 50,000,000 rows, kept 2,427,343
scanned 52,000,000 rows, 

## 5. Limitations

*What this work cannot claim.*
## 5. Limitations

This work cannot claim:
- **Causation.** It says where CTR is low for the position and tends to stay low. It does not show
  that changing a title or meta text raises clicks, and it does not say anything about how Google ranks.
- **Stability over time.** The data covers one period of about ____ weeks and has only ____ time
  windows, so there is a single time-aware split. Repeated client splits gave precision@50 of ____ ± ____ for the model and ____ ± ____ for the baseline.
- **A clean label.** CTR is noisy for low-volume pages: the label rate is ____ for the lowest-volume
  group and ____ for the highest. Part of every "persistent gap" is chance.
- **All pages.** The label is only defined for pages that still have at least 50 impressions in the
  next window, so pages that disappear from search are not judged.
- **Regression to the mean.** Some low-CTR pages recover by themselves, which makes any fix look
  better than it is.
- **What the table cannot see.** It has no information on queries, search features, AI answers or brand
  searches, any of which can lower CTR for reasons a title cannot fix.
- **Client mix.** ____ clients, so results may not carry over to other sites.

Language used in this paper: observed, directional, decision-support.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

def fit_predict(train_df, test_df):
    X_train = train_df[FEATURES]
    y_train = train_df["label"].values
    X_test = test_df[FEATURES]

    # Logistic Regression Pipeline
    lr = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=1000))
    lr.fit(X_train, y_train)

    # HistGradientBoostingClassifier
    hgb = HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0)
    hgb.fit(X_train, y_train)

    # Predictions (probabilities of class 1)
    preds = {
        "lr": lr.predict_proba(X_test)[:, 1],
        "boosting": hgb.predict_proba(X_test)[:, 1]
    }
    return preds, lr, hgb

def evaluate(test_df, scores_dict):
    """Compute precision at top 50 and AUC for each scoring method."""
    y_true = test_df["label"].values
    results = []
    for method, scores in scores_dict.items():
        # Sort descending by score
        idx = np.argsort(scores)[::-1]
        top_50_idx = idx[:50]
        p50 = y_true[top_50_idx].mean() if len(y_true) >= 50 else y_true.mean()
        try:
            auc = roc_auc_score(y_true, scores)
        except:
            auc = 0.5
        results.append({"method": method, "p@50": p50, "auc": auc})

    # Also ensure "test" variable is available globally for the rest of Section 5
    global test
    test = test_df

    return pd.DataFrame(results), test_df

In [ ]:
if GROUPED:
    rows = []
    for seed in range(1, 6):
        tr, te = get_split(seed)
        if tr["label"].nunique() < 2 or len(te) < 30:
            continue
        pr, _, _ = fit_predict(tr, te)
        sc = {"baseline": te["missing_clicks"].values,
              "boosting": pr["boosting"],
              "boosting_x_missing": pr["boosting"] * te["missing_clicks"].values}
        res, _ = evaluate(te, sc)
        for _, r in res.iterrows():
            rows.append({"seed": seed, "method": r["method"], "p50": r["p@50"], "auc": r["auc"]})
    if rows:
        rep = pd.DataFrame(rows)
        print("Repeated client splits (mean and spread of precision@50):")
        print(rep.groupby("method").agg(mean_p50=("p50", "mean"), sd_p50=("p50", "std"),
                                        mean_auc=("auc", "mean"), splits=("seed", "nunique")).round(3))
    else:
        print("No usable repeated splits.")
else:
    print("Skipped: the split was time-only, so there are no repeated client splits. Say this in the text.")

t = D_test_all.copy()
t["volume_group"] = pd.qcut(t["impr_l"].rank(method="first"), 4,
                            labels=["lowest", "low-mid", "high-mid", "highest"])
print("\nLabel rate by label-window volume (test pages):")
print(t.groupby("volume_group", observed=True).agg(n=("label", "size"),
                                                  label_rate=("label", "mean")).round(3))

Repeated client splits (mean and spread of precision@50):
                    mean_p50  sd_p50  mean_auc  splits
method                                                
baseline                0.90     0.0     0.655       5
boosting                0.86     0.0     0.690       5
boosting_x_missing      0.94     0.0     0.666       5

Label rate by label-window volume (test pages):
                n  label_rate
volume_group                 
lowest        701       0.676
low-mid       701       0.632
high-mid      700       0.581
highest       701       0.648


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*
## 6. Ranked recommendations

Pages are ranked by **priority = chance the CTR gap persists x clicks missing versus normal**,
using the most recent 14 days. Each page gets one reason code and one action label.

| Reason code | Action | Meaning |
|---|---|---|
| PERSISTENT_CTR_GAP | REVIEW_TITLE_META | CTR is well below normal for its position and the model expects it to stay low |
| CTR_GAP_MAY_RECOVER | MONITOR | CTR is below normal but the gap is likely to close by itself |
| HIGH_VOLUME_HEALTHY | PROTECT | many impressions and CTR is fine, so avoid risky changes |
| OK | NO_ACTION | nothing stands out |

(Paste the top-20 table here. Page references are short and non-identifying.)

These are review priorities, not predictions of what a change will do. A person should look at each page
before acting: a low CTR can come from a brand search, an AI answer, or a search feature, none of which a title edit fixes.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
import pandas as pd
import numpy as np

# 1. Train the final chosen Gradient Boosting model on all training data
clf = HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0)
clf.fit(D_train_all[FEATURES], D_train_all["label"].values)

# 2. Build features for the "now" window (the most recent 14 days of the base data)
now_end = base["report_date"].max()
now_start = now_end - pd.Timedelta(days=14)

Fnow, normal_now = build_dataset(now_start, now_end, None)

# 3. Predict persistence probability
Fnow["p_persist"] = clf.predict_proba(Fnow[FEATURES])[:, 1]

# 4. Compute priority: chance of persistence * missing clicks
Fnow["priority"] = Fnow["p_persist"] * Fnow["missing_clicks"]

# 5. Apply reason codes and actions
high_vol_threshold = Fnow["impr"].quantile(0.90)

conditions = [
    (Fnow["ctr_ratio"] < 0.8) & (Fnow["p_persist"] >= 0.5), # PERSISTENT_CTR_GAP
    (Fnow["ctr_ratio"] < 0.8) & (Fnow["p_persist"] < 0.5),  # CTR_GAP_MAY_RECOVER
    (Fnow["ctr_ratio"] >= 0.8) & (Fnow["impr"] >= high_vol_threshold) # HIGH_VOLUME_HEALTHY
]
codes = ["PERSISTENT_CTR_GAP", "CTR_GAP_MAY_RECOVER", "HIGH_VOLUME_HEALTHY"]

Fnow["reason_code"] = np.select(conditions, codes, default="OK")

action_map = {
    "PERSISTENT_CTR_GAP": "REVIEW_TITLE_META",
    "CTR_GAP_MAY_RECOVER": "MONITOR",
    "HIGH_VOLUME_HEALTHY": "PROTECT",
    "OK": "NO_ACTION"
}
Fnow["action"] = Fnow["reason_code"].map(action_map)

# 6. Extract top 20 recommendations ranked by priority
queue_now = Fnow.sort_values(by="priority", ascending=False).copy()
queue_now["page_id"] = queue_now["content_hash_id"].str[:6]
queue_now["client_id"] = queue_now["client_hash_id"].str[:6]

top_20 = queue_now[["client_id", "page_id", "impr", "ctr", "expected_ctr", "missing_clicks", "p_persist", "priority", "reason_code", "action"]].head(20)

display(top_20.round(3))

,client_id,page_id,impr,ctr,expected_ctr,missing_clicks,p_persist,priority,reason_code,action
7978,client,conten,16988,0.002,0.005,52.412,0.817,42.838,PERSISTENT_CTR_GAP,REVIEW_TITLE_META
13124,client,conten,10963,0.001,0.005,35.893,0.824,29.592,PERSISTENT_CTR_GAP,REVIEW_TITLE_META
5408,client,conten,10132,0.001,0.005,32.960,0.887,29.227,PERSISTENT_CTR_GAP,REVIEW_TITLE_META
496,client,conten,9894,0.001,0.005,33.833,0.824,27.894,PERSISTENT_CTR_GAP,REVIEW_TITLE_META
12571,client,conten,8062,0.003,0.007,32.151,0.808,25.993,PERSISTENT_CTR_GAP,REVIEW_TITLE_META
2746,client,conten,10395,0.002,0.005,31.205,0.817,25.504,PERSISTENT_CTR_GAP,REVIEW_TITLE_META
12528,client,conten,7728,0.000,0.005,35.580,0.700,24.921,PERSISTENT_CTR_GAP,REVIEW_TITLE_META
2799,client,conten,7553,0.000,0.005,33.752,0.679,22.926,PERSISTENT_CTR_GAP,REVIEW_TITLE_META
12584,client,conten,7302,0.001,0.005,27.564,0.815,22.469,PERSISTENT_CTR_GAP,REVIEW_TITLE_META
10509,client,conten,7307,0.000,0.005,31.588,0.701,22.156,PERSISTENT_CTR_GAP,REVIEW_TITLE_META


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*
## 7. Artifacts the paper embeds

- `fig1_precision_at_50.png`: model vs baseline, the main result
- `fig2_normal_ctr_by_position.png`: how "normal CTR" was defined
- `fig3_reason_codes.png`: how pages split across reason codes
- `fig4_top20_table.png`: the ranked recommendations, with short non-identifying page references
- `capstone_metrics.json`: the run's numbers, used for every figure quoted in the text

In [ ]:
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Ensure output directories exist
os.makedirs("work/outputs/figures", exist_ok=True)

sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 11, 'axes.labelsize': 12, 'axes.titlesize': 13})

# -----------------------------------------------------
# Artifact 1: fig1_precision_at_50.png
# -----------------------------------------------------
fig1, ax1 = plt.subplots(figsize=(7, 4.5))
methods = ['Baseline', 'Boosting (Model)', 'Boosting x Missing (Hybrid)']
precisions = [0.90, 0.86, 0.94]
colors = ['#b0bec5', '#1976d2', '#2e7d32']

bars = ax1.bar(methods, precisions, color=colors, width=0.5, edgecolor='black', alpha=0.9)
ax1.set_ylim(0, 1.15)
ax1.set_ylabel("Precision @ 50")
ax1.set_title("Figure 1: Comparison of Precision @ 50")
for bar in bars:
    yval = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2.0, yval + 0.02, f"{yval:.2%}", ha='center', va='bottom', weight='bold')

plt.tight_layout()
plt.savefig("work/outputs/figures/fig1_precision_at_50.png", dpi=150)
plt.close(fig1)

# -----------------------------------------------------
# Artifact 2: fig2_normal_ctr_by_position.png
# -----------------------------------------------------
fig2, ax2 = plt.subplots(figsize=(7, 4.5))
bands = list(normal_now.keys())
ctrs = list(normal_now.values())

# Sort bands for correct rank order
sorted_indices = np.argsort([POS_LABELS.index(b) for b in bands])
sorted_bands = [bands[i] for i in sorted_indices]
sorted_ctrs = [ctrs[i] for i in sorted_indices]

ax2.plot(sorted_bands, sorted_ctrs, marker='o', linewidth=2.5, color='#d32f2f', markersize=8)
ax2.set_ylabel("Normal CTR (Pooled)")
ax2.set_xlabel("Position Band")
ax2.set_title("Figure 2: Normal CTR by Position Band (Current Window)")
for i, ctr in enumerate(sorted_ctrs):
    ax2.text(i, ctr + 0.0005, f"{ctr:.3%}", ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig("work/outputs/figures/fig2_normal_ctr_by_position.png", dpi=150)
plt.close(fig2)

# -----------------------------------------------------
# Artifact 3: fig3_reason_codes.png
# -----------------------------------------------------
fig3, ax3 = plt.subplots(figsize=(7, 4.5))
counts = Fnow["reason_code"].value_counts()

counts.plot(kind='bar', color='#f57c00', edgecolor='black', alpha=0.85, ax=ax3)
ax3.set_ylabel("Number of Pages")
ax3.set_xlabel("Reason Code")
ax3.set_title("Figure 3: Breakdown of Pages by Reason Code")
plt.xticks(rotation=15, ha='right')

for i, count in enumerate(counts):
    ax3.text(i, count + (max(counts) * 0.01), f"{count:,}", ha='center', va='bottom', weight='bold')

plt.tight_layout()
plt.savefig("work/outputs/figures/fig3_reason_codes.png", dpi=150)
plt.close(fig3)

# -----------------------------------------------------
# Artifact 4: fig4_top20_table.png
# -----------------------------------------------------
fig4, ax4 = plt.subplots(figsize=(10, 6))
ax4.axis('off')
table_df = top_20.copy().round(4)
# Rename columns for clarity
table_df.columns = ['Client', 'Page', 'Impr', 'CTR', 'Expected', 'Missing Clicks', 'P(Persist)', 'Priority', 'Reason', 'Action']

the_table = ax4.table(cellText=table_df.values, colLabels=table_df.columns, loc='center', cellLoc='center')
the_table.auto_set_font_size(False)
the_table.set_fontsize(8)
the_table.scale(1.1, 1.4)

# Header coloring
for k, cell in the_table.get_celld().items():
    if k[0] == 0:
        cell.set_text_props(weight='bold', color='white')
        cell.set_facecolor('#0d47a1')

ax4.set_title("Figure 4: Top 20 Ranked Recommendations Table Preview", pad=20, weight='bold')
plt.savefig("work/outputs/figures/fig4_top20_table.png", dpi=150, bbox_inches='tight')
plt.close(fig4)

# -----------------------------------------------------
# Artifact 5: capstone_metrics.json
# -----------------------------------------------------
metrics = {
    "metadata": {
        "scanned_rows": int(seen),
        "kept_rows": len(df),
        "unique_clients": int(df["client_hash_id"].nunique()),
        "days_of_data": int(span)
    },
    "evaluation": {
        "baseline_precision_at_50": 0.90,
        "boosting_precision_at_50": 0.86,
        "hybrid_precision_at_50": 0.94
    },
    "segments": {
        "lowest_volume_label_rate": 0.676,
        "highest_volume_label_rate": 0.648
    }
}

with open("work/outputs/capstone_metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

print("SUCCESS: Created fig1_precision_at_50.png, fig2_normal_ctr_by_position.png, fig3_reason_codes.png, fig4_top20_table.png, and capstone_metrics.json in their respective paths.")

SUCCESS: Created fig1_precision_at_50.png, fig2_normal_ctr_by_position.png, fig3_reason_codes.png, fig4_top20_table.png, and capstone_metrics.json in their respective paths.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
